In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece

In the previous models, we used TF-IDF, a Multi-Layer Perceptron (MLP) built from scratch, and BGE embeddings with an MLP to solve the Smart MCQ Solver Challenge. Although these models produced good results, they relied on manually designed features or sentence embeddings.

In this model, we use RoBERTa (Robustly Optimized BERT Approach), a pretrained transformer model developed by Meta AI. Unlike the previous models, RoBERTa learns the relationship between the question and each answer option directly from the text, allowing it to understand the meaning and context more effectively.

The multiple-choice problem is converted into a pairwise binary classification task. For every question, five question–option pairs are created. The correct option is labeled as 1, while the remaining four options are labeled as 0. RoBERTa is then fine-tuned to predict the probability that each option is the correct answer. During inference, the option with the highest probability is selected as the final answer, and the top three probabilities are used to generate the Top-3 predictions for calculating the MAP@3 score.

In [ ]:
# Importing libraries
# ---------------------

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import wandb

from kaggle_secrets import UserSecretsClient

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback
)

from datasets import Dataset

In [ ]:
# Weights & Biases Login
# ------------------------

user_secrets = UserSecretsClient()

api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=api_key)

wandb.init(

    entity="25ds1000066-dl-genai-project",

    project="25ds1000066-t22026",

    name="Model5_RoBERTa",

    config={

        "model":"RoBERTa-base",

        "architecture":"Pairwise Classification",

        "optimizer":"AdamW",

        "learning_rate":2e-5,

        "epochs":5

    }

)

In [ ]:
# Configuration
# ---------------

MODEL_NAME = "roberta-base"              # Pretrained RoBERTa model

MAX_LENGTH = 320                                      # Maximum number of tokens for each question-option pair

BATCH_SIZE = 8                                       # Number of samples processed in one batch

LR = 2e-5                                            # Learning rate for AdamW optimizer

EPOCHS = 5

SEED = 42                                            # Random seed for reproducibility


# Set Random Seed
# -----------------

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)


# Select Training Device
# -------------------------

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print(device)

In [ ]:
# Reading the Dataset

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

In [ ]:
# Split Dataset into Training and Validation Sets
# -------------------------------------------------

from sklearn.model_selection import train_test_split

# Split the original training data into
# 80% training and 20% validation

train_split, valid_split = train_test_split(
    train,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Training Questions :", len(train_split))
print("Validation Questions :", len(valid_split))

In [ ]:
# Create Pairwise Training Dataset
# ---------------------------------

# Convert each question into five question-option pairs.
# The correct option receives label 1, and the remaining
# four options receive label 0.

def create_pairwise_dataset(dataframe):

    pairwise_data = []

    for _, row in dataframe.iterrows():

        prompt = str(row["prompt"])

        correct_answer = row["answer"]

        # Create one training example for each option
        for option in ["A", "B", "C", "D", "E"]:

            pairwise_data.append({

                "question": prompt,

                "option": str(row[option]),

                # Label = 1 for the correct answer
                # Label = 0 for incorrect answers
                "label": int(option == correct_answer),

                # Store for later evaluation
                "correct_option": correct_answer,
                "option_name": option

            })

    return pd.DataFrame(pairwise_data)


# Create Pairwise Training and Validation Data
# ----------------------------------------------

roberta_train_pairwise = create_pairwise_dataset(train_split)

roberta_valid_pairwise = create_pairwise_dataset(valid_split)

print("Training Samples :", len(roberta_train_pairwise))

print("Validation Samples :", len(roberta_valid_pairwise))

display(roberta_train_pairwise.head())

In [ ]:
# Load RoBERTa Tokenizer
#--------------------------

# Load the tokenizer corresponding to the pretrained
# DeBERTa model.

roberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded Successfully")

In [ ]:
# Tokenize Question-Option Pairs
# --------------------------------

# Convert each question-option pair into token IDs
# and attention masks that can be fed into RoBERTa.

def tokenize_function(examples):

    return roberta_tokenizer(

        examples["question"],
        examples["option"],

        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH

    )

In [ ]:
# Convert Pandas DataFrames to Hugging Face Datasets
#-----------------------------------------------------

roberta_train_dataset = Dataset.from_pandas(
    roberta_train_pairwise
)

roberta_valid_dataset = Dataset.from_pandas(
    roberta_valid_pairwise
)

In [ ]:
# Apply Tokenization
# --------------------

roberta_train_dataset = roberta_train_dataset.map(
    tokenize_function,
    batched=True
)

roberta_valid_dataset = roberta_valid_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
# Rename Label Column
# ---------------------

roberta_train_dataset = roberta_train_dataset.rename_column(
    "label",
    "labels"
)

roberta_valid_dataset = roberta_valid_dataset.rename_column(
    "label",
    "labels"
)


# Keep Only Required Columns
# ---------------------------

roberta_train_dataset.set_format(

    type="torch",

    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]

)

roberta_valid_dataset.set_format(

    type="torch",

    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]

)

print(roberta_train_dataset)
print(roberta_valid_dataset)

In [ ]:
# Load Pretrained RoBERTa Model
# ------------------------------

# Binary classifier
# Label 1 = Correct Answer
# Label 0 = Incorrect Answer

roberta_model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2

)

roberta_model.to(device)

print("RoBERTa Loaded Successfully")

In [ ]:
# Compute Evaluation Metrics
# ----------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {

        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1

    }

In [ ]:
# For correct W&B login metrice
# ------------------------------

class WandbMetricsCallback(TrainerCallback):

    def on_log(self, args, state, control, logs=None, **kwargs):

        if logs is None:
            return

        metrics = {}

        # Training Loss
        if "loss" in logs:
            metrics["Training Loss"] = logs["loss"]

        # Validation Loss
        if "eval_loss" in logs:
            metrics["Validation Loss"] = logs["eval_loss"]

        # Validation Accuracy
        if "eval_accuracy" in logs:
            metrics["Validation Accuracy"] = logs["eval_accuracy"]

        if metrics:
            wandb.log(metrics)

In [ ]:
# Training Arguments
# -------------------

# Configure the Hugging Face Trainer

training_args = TrainingArguments(

    # Folder to save checkpoints
    output_dir="./roberta_checkpoints",

    # Training and Validation
    do_train=True,
    do_eval=True,

    # Evaluate after every epoch
    eval_strategy="epoch",

    # Save checkpoint after every epoch
    save_strategy="epoch",

    # Load the best checkpoint automatically
    load_best_model_at_end=True,

    # Use Validation F1 to determine the best model
    metric_for_best_model="f1",

    greater_is_better=True,

    # Keep only the best checkpoint
    save_total_limit=1,

    # Batch size
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    # Number of epochs
    num_train_epochs=EPOCHS,

    # Learning rate
    learning_rate=LR,

    # Weight decay
    weight_decay=0.01,

    # Logging
    logging_strategy="steps",
    logging_steps=50,

    # Report metrics to Weights & Biases
    report_to="wandb",

    run_name="Model5_RoBERTa",

    # Reproducibility
    seed=SEED,

    # Remove unnecessary columns automatically
    remove_unused_columns=True,

   
    # Disabled mixed precision because it caused
    # FP16 gradient errors in the environment.
    fp16=False,
    bf16=False

)

In [ ]:
# Create Hugging Face Trainer
# -----------------------------

trainer = Trainer(

    model=roberta_model,

    args=training_args,

    train_dataset=roberta_train_dataset,

    eval_dataset=roberta_valid_dataset,

    compute_metrics=compute_metrics,

    # Stop training if Validation F1 does not improve
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        ),
        WandbMetricsCallback()
    ]

)

In [ ]:
# Train the RoBERTa Model
# =====================================================

print("Starting RoBERTa Training...\n")

# Train the model
trainer.train()

print("\nTraining Completed Successfully!")

In [ ]:
# Generate Predictions on Validation Dataset
# --------------------------------------------

# Predict on the validation dataset

roberta_validation_output = trainer.predict(
    roberta_valid_dataset
)

# Extract logits

roberta_validation_logits = roberta_validation_output.predictions

print("Validation prediction shape:", roberta_validation_logits.shape)

In [ ]:
# Convert Logits to Probabilities
# ---------------------------------

import torch
import torch.nn.functional as F

# Convert logits to probabilities

roberta_validation_probabilities = F.softmax(

    torch.tensor(roberta_validation_logits),

    dim=1

).numpy()

print(roberta_validation_probabilities[:5])

In [ ]:
# Convert Pairwise Predictions to Question Predictions
# ------------------------------------------------------

roberta_validation_top1_predictions = []

roberta_validation_top3_predictions = []

for i in range(0, len(roberta_valid_pairwise), 5):

    # Probability that the option is the correct answer
    scores = roberta_validation_probabilities[i:i+5, 1]

    option_names = roberta_valid_pairwise.iloc[i:i+5]["option_name"].tolist()

    ranking = sorted(

        zip(option_names, scores),

        key=lambda x: x[1],

        reverse=True

    )

    roberta_validation_top1_predictions.append(

        ranking[0][0]

    )

    roberta_validation_top3_predictions.append(

        [x[0] for x in ranking[:3]]

    )

print(roberta_validation_top3_predictions[:5])

In [ ]:
# Creating MAP@3 Function

# Function to calculate Average Precision at K (AP@K) for a single prediction
def apk(actual, predicted, k=3):

    if len(predicted) > k:                   # Keep only the top-k predictions
        predicted = predicted[:k]            

    for i, p in enumerate(predicted):        # Iterate through the predicted labels
        if p == actual:                      # If the correct answer is found, return the score
            return 1 / (i + 1)

    return 0                                 # Return 0 if the correct answer is not in the top-k predictions


# Function to calculate Mean Average Precision at K (MAP@K)
def mapk(actuals, predictions, k=3):
    
    # Compute the average AP@K score over the entire dataset
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

In [ ]:
# Calculate Validation Metrics
# ------------------------------

# Actual correct answers for the validation questions

roberta_actual_answers = valid_split["answer"].tolist()


# Accuracy

roberta_accuracy = accuracy_score(

    roberta_actual_answers,

    roberta_validation_top1_predictions

)


# Precision

roberta_precision = precision_score(

    roberta_actual_answers,

    roberta_validation_top1_predictions,

    average="macro"

)


# Recall

roberta_recall = recall_score(

    roberta_actual_answers,

    roberta_validation_top1_predictions,

    average="macro"

)


# F1 Score

roberta_f1 = f1_score(

    roberta_actual_answers,

    roberta_validation_top1_predictions,

    average="macro"

)


# MAP@3

roberta_map3 = mapk(

    roberta_actual_answers,

    roberta_validation_top3_predictions

)


# Print Results

print(f"Accuracy : {roberta_accuracy:.4f}")

print(f"Precision : {roberta_precision:.4f}")

print(f"Recall : {roberta_recall:.4f}")

print(f"F1 Score : {roberta_f1:.4f}")

print(f"MAP@3 : {roberta_map3:.4f}")

In [ ]:
# Log Final Validation Metrics to Weights & Biases
# -------------------------------------------------

wandb.log({

    "Accuracy": roberta_accuracy,

    "Precision": roberta_precision,

    "Recall": roberta_recall,

    "F1 Score": roberta_f1,

    "MAP@3": roberta_map3,

    "Kaggle Score": 0.73233

})

In [ ]:
# Save the Fine-tuned RoBERTa Model
# -----------------------------------

SAVE_PATH = "/kaggle/working/roberta_model"

trainer.save_model(SAVE_PATH)

print("RoBERTa model saved successfully.")

In [ ]:
# Save the Tokenizer
# --------------------

roberta_tokenizer.save_pretrained(SAVE_PATH)

print("Tokenizer saved successfully.")

In [ ]:
# Save Validation Metrics
# ----------------------------

import json

roberta_metrics = {

    "Accuracy": float(roberta_accuracy),

    "Precision": float(roberta_precision),

    "Recall": float(roberta_recall),

    "F1 Score": float(roberta_f1),

    "MAP@3": float(roberta_map3)

}

with open(

    f"{SAVE_PATH}/metrics.json",

    "w"

) as file:

    json.dump(

        roberta_metrics,

        file,

        indent=4

    )

print("Metrics saved successfully.")

In [ ]:
# Finish Weights & Biases Run
# ----------------------------

wandb.finish()